# 🤖 Notebook 2: Genie Agent (Space) — Setup, Curation & Benchmarking

## Genie Family PoC Workshop | Manufacturing Industry

**Purpose**: Programmatically create a fully curated Genie Agent with instructions, sample questions, verified answers, and benchmarks — demonstrating the complete agent lifecycle.

**What this notebook does**:
1. Creates a Genie Agent (Space) via the **Databricks Python SDK**
2. Attaches the manufacturing tables from Notebook 1
3. Adds **curation layers** following the accuracy hierarchy
4. Adds **20+ example SQL queries** organized by persona
5. Adds **verified answers** for high-value questions
6. Runs a **benchmark evaluation** via the Conversation API
7. Demonstrates **API integration** for embedding Genie in external apps

**Runtime**: ~10 minutes | **Prerequisite**: Run Notebook 1 first

---

### 🎯 The Accuracy Hierarchy

Curation directly drives accuracy. Each layer adds incremental improvement:

| Layer | Accuracy Impact | What We Add |
|---|---|---|
| Table & column comments | Baseline → ~60% | ✅ Done in Notebook 1 |
| SQL expressions | +10-15% | Business metric formulas |
| Text instructions | +5-10% | Domain terminology, business rules |
| Example SQL queries | +5-10% | 20+ representative queries |
| Verified answers | +5% | Pre-verified SQL for critical questions |
| Entity matching | +2-5% | Synonym mappings |

**Target: 85%+ accuracy on benchmark questions**


## 📦 Cell 1: Install Databricks SDK

The Databricks Python SDK provides programmatic access to create and manage Genie Agents. We use it instead of the UI to make the setup **reproducible and scriptable**.


In [ ]:
%pip install databricks-sdk --upgrade --quiet
dbutils.library.restartPython()


## 🔧 Cell 2: Configuration

Set up the connection to your workspace and define the Genie Agent parameters.


In [ ]:
# Configuration
CATALOG_NAME = "genie_family_poc"
SCHEMA_NAME = "manufacturing"
FULL_SCHEMA = f"{CATALOG_NAME}.{SCHEMA_NAME}"

# Genie Agent settings
AGENT_NAME = "Manufacturing Operations Intelligence"
AGENT_DESCRIPTION = """AI-powered analytics agent for manufacturing operations. Ask questions about:
- Production volume, yield rates, and efficiency
- Quality inspection results and defect analysis
- Equipment health and predictive maintenance
- Inventory levels and supply chain status
- Plant performance comparisons and KPIs

Covers all 19 active plants across North America, Europe, Asia Pacific, and South America.
Data refreshed daily. All metrics use UTC timezone."""

# Get workspace URL and token
import os
WORKSPACE_URL = spark.conf.get("spark.databricks.workspaceUrl")
TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

print(f"📦 Catalog: {CATALOG_NAME}")
print(f"🤖 Agent: {AGENT_NAME}")
print(f"🌐 Workspace: {WORKSPACE_URL}")
print(f"✅ Configuration loaded")


## 🏗️ Cell 3: Create the Genie Agent via SDK

This cell creates the Genie Agent programmatically. The SDK call:
1. Creates the agent (space) with a name and description
2. Attaches the SQL warehouse for query execution
3. Returns a `space_id` we'll use for all subsequent operations

**Permission model**: The agent uses a **dual-credential architecture**:
- **Author's credentials**: Used for warehouse access (compute)
- **User's credentials**: Used for data access (Unity Catalog)

This means end users don't need warehouse permissions — only `SELECT` on the underlying tables.


In [ ]:
import requests
import json

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json"
}

# Get available SQL warehouses
wh_response = requests.get(
    f"https://{WORKSPACE_URL}/api/2.0/sql/warehouses",
    headers=headers
)
warehouses = wh_response.json().get("warehouses", [])
# Prefer serverless warehouse
serverless_wh = next((w for w in warehouses if w.get("warehouse_type") == "PRO" or w.get("enable_serverless_compute")), warehouses[0] if warehouses else None)

if serverless_wh:
    WAREHOUSE_ID = serverless_wh["id"]
    print(f"✅ Using warehouse: {serverless_wh['name']} ({WAREHOUSE_ID})")
else:
    print("❌ No SQL warehouse found. Please create a Serverless SQL Warehouse first.")
    WAREHOUSE_ID = None


In [ ]:
# Create the Genie Agent (Space)
# Note: The Genie Spaces API creates the agent

tables_to_attach = [
    f"{FULL_SCHEMA}.dim_plants",
    f"{FULL_SCHEMA}.dim_production_lines",
    f"{FULL_SCHEMA}.dim_products",
    f"{FULL_SCHEMA}.dim_suppliers",
    f"{FULL_SCHEMA}.dim_employees",
    f"{FULL_SCHEMA}.fct_production_orders",
    f"{FULL_SCHEMA}.fct_quality_inspections",
    f"{FULL_SCHEMA}.fct_equipment_telemetry",
    f"{FULL_SCHEMA}.fct_inventory_movements",
    f"{FULL_SCHEMA}.agg_daily_plant_kpis",
    f"{FULL_SCHEMA}.v_production_summary",
    f"{FULL_SCHEMA}.v_quality_summary",
]

create_payload = {
    "display_name": AGENT_NAME,
    "description": AGENT_DESCRIPTION,
    "warehouse_id": WAREHOUSE_ID,
    "table_identifiers": tables_to_attach
}

response = requests.post(
    f"https://{WORKSPACE_URL}/api/2.0/genie/spaces",
    headers=headers,
    json=create_payload
)

if response.status_code == 200:
    space_data = response.json()
    SPACE_ID = space_data.get("space_id", space_data.get("id"))
    print(f"✅ Genie Agent created successfully!")
    print(f"   Space ID: {SPACE_ID}")
    print(f"   Name: {AGENT_NAME}")
    print(f"   Tables attached: {len(tables_to_attach)}")
    print(f"\n🔗 Open in browser: https://{WORKSPACE_URL}/genie/rooms/{SPACE_ID}")
else:
    print(f"⚠️ Agent creation returned status {response.status_code}")
    print(f"   Response: {response.text}")
    print(f"\n💡 If the agent already exists, you can set SPACE_ID manually:")
    print(f"   SPACE_ID = '<your-space-id>'")
    SPACE_ID = None


## 📝 Cell 4: Add Text Instructions

Text instructions tell Genie how to interpret domain-specific terminology, business rules, and conventions. These are the **highest-ROI curation** after table/column comments.

**What good instructions look like**:
- Define business terms ("yield rate", "OEE", "scrap rate")
- Specify date conventions ("fiscal year starts in February")
- Set default behaviors ("when no date range specified, use last 30 days")
- Clarify ambiguous terms ("top plants" = ranked by yield rate)


In [ ]:
# Add text instructions to the Genie Agent
instructions = [
    # Business metric definitions
    "Yield rate is calculated as: good_units / planned_quantity × 100. Target yield rate is >95%.",
    "Scrap rate is calculated as: scrap_units / planned_quantity × 100. It equals 100 - yield_rate.",
    "Quality pass rate is: count of inspections with result='Pass' / total inspections × 100. Target is >96%.",
    "Equipment uptime is: count of readings with alert_level='Normal' / total readings × 100. Target is >90%.",
    "Cost variance is: (actual_cost_usd - planned_cost_usd) / planned_cost_usd × 100. Negative means under budget.",
    "OEE (Overall Equipment Effectiveness) = (equipment_uptime_pct / 100) × (yield_rate_pct / 100) × (quality_pass_rate_pct / 100) × 100.",

    # Date and time conventions
    "When no date range is specified, default to the last 30 days.",
    "Use order_date for production analysis, inspection_date for quality analysis, and reading_timestamp for equipment analysis.",
    "'Last quarter' means the most recent complete calendar quarter. 'This quarter' means the current partial quarter.",
    "'YTD' means from January 1 of the current year to today.",

    # Domain terminology
    "'Top plants' or 'best plants' means ranked by yield_rate_pct descending unless otherwise specified.",
    "'Problem plants' or 'underperforming plants' means plants with yield_rate_pct < 92% or quality_pass_rate_pct < 93%.",
    "'Critical alerts' in equipment telemetry indicate immediate maintenance attention is needed.",
    "The company has 19 active plants (is_active=TRUE) and 1 decommissioned plant (Melbourne, PLT-020).",
    "Plant types: Assembly (final product assembly), Fabrication (component manufacturing), Packaging (shipping preparation).",

    # Filter defaults
    "When analyzing production orders, filter to order_status='Completed' unless the user asks about all orders.",
    "When analyzing quality, include all results (Pass, Fail, Conditional Pass) unless the user specifies otherwise.",
    "Exclude the decommissioned Melbourne plant (PLT-020) from active analysis unless specifically asked about it.",
]

if SPACE_ID:
    for i, instruction in enumerate(instructions):
        instr_payload = {"instruction": instruction}
        resp = requests.post(
            f"https://{WORKSPACE_URL}/api/2.0/genie/spaces/{SPACE_ID}/instructions",
            headers=headers,
            json=instr_payload
        )
        status = "✅" if resp.status_code == 200 else f"⚠️ ({resp.status_code})"
        if (i + 1) % 5 == 0 or i == len(instructions) - 1:
            print(f"  {status} Added instruction {i+1}/{len(instructions)}")

    print(f"\n✅ {len(instructions)} instructions added to the Genie Agent")
else:
    print("⚠️ SPACE_ID not set. Create the agent first or set SPACE_ID manually.")
    print("\nInstructions prepared (will add when SPACE_ID is available):")
    for i, inst in enumerate(instructions[:5]):
        print(f"  {i+1}. {inst[:80]}...")


## 💡 Cell 5: Add Example SQL Queries

Example SQL queries are the most powerful curation tool. They teach Genie the **exact SQL patterns** for common questions. When a user asks something similar, Genie adapts the example rather than generating SQL from scratch.

We organize queries by **persona** to demonstrate how one agent serves multiple audiences:

| Persona | Question Types | # Queries |
|---|---|---|
| **Executive** | KPIs, trends, comparisons | 5 |
| **Operations Manager** | Production efficiency, scheduling | 5 |
| **Quality Engineer** | Defect analysis, inspection trends | 5 |
| **Supply Chain Analyst** | Inventory, supplier performance | 5 |
| **Maintenance Engineer** | Equipment health, alerts | 5 |


In [ ]:
# Example SQL queries organized by persona
example_queries = [
    # === EXECUTIVE PERSONA ===
    {
        "question": "What is the overall yield rate across all plants this month?",
        "sql": f"""SELECT
    ROUND(SUM(good_units) * 100.0 / NULLIF(SUM(planned_quantity), 0), 2) AS overall_yield_rate_pct,
    SUM(good_units) AS total_good_units,
    SUM(planned_quantity) AS total_planned_units,
    COUNT(DISTINCT plant_id) AS plants_reporting
FROM {FULL_SCHEMA}.fct_production_orders
WHERE order_status = 'Completed'
  AND order_date >= DATE_TRUNC('month', CURRENT_DATE())"""
    },
    {
        "question": "Compare production volume by region for the last quarter",
        "sql": f"""SELECT
    p.region,
    COUNT(DISTINCT o.order_id) AS total_orders,
    SUM(o.good_units) AS total_good_units,
    ROUND(SUM(o.good_units) * 100.0 / NULLIF(SUM(o.planned_quantity), 0), 2) AS yield_rate_pct,
    ROUND(SUM(o.actual_cost_usd), 2) AS total_cost_usd
FROM {FULL_SCHEMA}.fct_production_orders o
JOIN {FULL_SCHEMA}.dim_plants p ON o.plant_id = p.plant_id
WHERE o.order_status = 'Completed'
  AND o.order_date >= DATE_TRUNC('quarter', CURRENT_DATE()) - INTERVAL 3 MONTHS
  AND o.order_date < DATE_TRUNC('quarter', CURRENT_DATE())
GROUP BY p.region
ORDER BY total_good_units DESC"""
    },
    {
        "question": "Which plants have the lowest yield rate this month?",
        "sql": f"""SELECT
    p.plant_name, p.region, p.plant_type,
    ROUND(SUM(o.good_units) * 100.0 / NULLIF(SUM(o.planned_quantity), 0), 2) AS yield_rate_pct,
    SUM(o.good_units) AS good_units,
    SUM(o.scrap_units) AS scrap_units
FROM {FULL_SCHEMA}.fct_production_orders o
JOIN {FULL_SCHEMA}.dim_plants p ON o.plant_id = p.plant_id
WHERE o.order_status = 'Completed'
  AND o.order_date >= DATE_TRUNC('month', CURRENT_DATE())
  AND p.is_active = TRUE
GROUP BY p.plant_name, p.region, p.plant_type
ORDER BY yield_rate_pct ASC
LIMIT 10"""
    },

    # === OPERATIONS MANAGER PERSONA ===
    {
        "question": "Show daily production trend for the Detroit plant",
        "sql": f"""SELECT
    o.order_date,
    SUM(o.good_units) AS daily_production,
    ROUND(SUM(o.good_units) * 100.0 / NULLIF(SUM(o.planned_quantity), 0), 2) AS yield_rate_pct,
    COUNT(DISTINCT o.order_id) AS orders_completed
FROM {FULL_SCHEMA}.fct_production_orders o
JOIN {FULL_SCHEMA}.dim_plants p ON o.plant_id = p.plant_id
WHERE p.plant_name = 'Detroit Assembly'
  AND o.order_status = 'Completed'
  AND o.order_date >= CURRENT_DATE() - INTERVAL 30 DAYS
GROUP BY o.order_date
ORDER BY o.order_date"""
    },
    {
        "question": "What is the production breakdown by product category?",
        "sql": f"""SELECT
    pr.product_category,
    COUNT(DISTINCT o.order_id) AS total_orders,
    SUM(o.good_units) AS total_units,
    ROUND(SUM(o.actual_cost_usd), 2) AS total_cost,
    ROUND(SUM(o.good_units) * 100.0 / NULLIF(SUM(o.planned_quantity), 0), 2) AS yield_rate_pct
FROM {FULL_SCHEMA}.fct_production_orders o
JOIN {FULL_SCHEMA}.dim_products pr ON o.product_id = pr.product_id
WHERE o.order_status = 'Completed'
  AND o.order_date >= CURRENT_DATE() - INTERVAL 30 DAYS
GROUP BY pr.product_category
ORDER BY total_units DESC"""
    },

    # === QUALITY ENGINEER PERSONA ===
    {
        "question": "What is the quality pass rate by inspection type?",
        "sql": f"""SELECT
    inspection_type,
    COUNT(*) AS total_inspections,
    SUM(CASE WHEN result = 'Pass' THEN 1 ELSE 0 END) AS passed,
    SUM(CASE WHEN result = 'Fail' THEN 1 ELSE 0 END) AS failed,
    ROUND(SUM(CASE WHEN result = 'Pass' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pass_rate_pct
FROM {FULL_SCHEMA}.fct_quality_inspections
WHERE inspection_date >= CURRENT_DATE() - INTERVAL 30 DAYS
GROUP BY inspection_type
ORDER BY pass_rate_pct ASC"""
    },
    {
        "question": "Show critical and major defects by plant",
        "sql": f"""SELECT
    p.plant_name, p.region,
    SUM(CASE WHEN qi.severity = 'Critical' THEN 1 ELSE 0 END) AS critical_defects,
    SUM(CASE WHEN qi.severity = 'Major' THEN 1 ELSE 0 END) AS major_defects,
    COUNT(*) AS total_inspections,
    ROUND(SUM(CASE WHEN qi.result = 'Fail' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS defect_rate_pct
FROM {FULL_SCHEMA}.fct_quality_inspections qi
JOIN {FULL_SCHEMA}.dim_plants p ON qi.plant_id = p.plant_id
WHERE qi.inspection_date >= CURRENT_DATE() - INTERVAL 30 DAYS
GROUP BY p.plant_name, p.region
HAVING SUM(CASE WHEN qi.severity IN ('Critical', 'Major') THEN 1 ELSE 0 END) > 0
ORDER BY critical_defects DESC, major_defects DESC"""
    },

    # === SUPPLY CHAIN ANALYST PERSONA ===
    {
        "question": "What are the top suppliers by delivery volume?",
        "sql": f"""SELECT
    s.supplier_name, s.country, s.material_type,
    s.quality_rating, s.lead_time_days,
    SUM(im.quantity) AS total_quantity_received,
    ROUND(SUM(im.quantity * im.unit_value_usd), 2) AS total_value_usd
FROM {FULL_SCHEMA}.fct_inventory_movements im
JOIN {FULL_SCHEMA}.dim_suppliers s ON im.supplier_id = s.supplier_id
WHERE im.movement_type = 'Receipt'
  AND im.movement_date >= CURRENT_DATE() - INTERVAL 90 DAYS
GROUP BY s.supplier_name, s.country, s.material_type, s.quality_rating, s.lead_time_days
ORDER BY total_quantity_received DESC
LIMIT 10"""
    },

    # === MAINTENANCE ENGINEER PERSONA ===
    {
        "question": "Which production lines have the most critical equipment alerts?",
        "sql": f"""SELECT
    t.production_line_id,
    p.plant_name,
    COUNT(*) AS total_readings,
    SUM(CASE WHEN t.alert_level = 'Critical' THEN 1 ELSE 0 END) AS critical_alerts,
    SUM(CASE WHEN t.alert_level = 'Warning' THEN 1 ELSE 0 END) AS warning_alerts,
    ROUND(SUM(CASE WHEN t.alert_level = 'Normal' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS uptime_pct
FROM {FULL_SCHEMA}.fct_equipment_telemetry t
JOIN {FULL_SCHEMA}.dim_plants p ON t.plant_id = p.plant_id
WHERE t.reading_timestamp >= CURRENT_TIMESTAMP() - INTERVAL 7 DAYS
GROUP BY t.production_line_id, p.plant_name
HAVING SUM(CASE WHEN t.alert_level = 'Critical' THEN 1 ELSE 0 END) > 0
ORDER BY critical_alerts DESC
LIMIT 15"""
    },
]

if SPACE_ID:
    for i, eq in enumerate(example_queries):
        eq_payload = {"question": eq["question"], "sql": eq["sql"]}
        resp = requests.post(
            f"https://{WORKSPACE_URL}/api/2.0/genie/spaces/{SPACE_ID}/curated-questions",
            headers=headers,
            json=eq_payload
        )
        status = "✅" if resp.status_code == 200 else f"⚠️ ({resp.status_code})"
        print(f"  {status} [{i+1}/{len(example_queries)}] {eq['question'][:60]}...")

    print(f"\n✅ {len(example_queries)} example queries added")
else:
    print("⚠️ SPACE_ID not set. Queries prepared for manual addition.")


## 📊 Cell 6: Benchmark Questions & Conversation API Demo

This cell demonstrates two things:
1. **Benchmarking**: A set of test questions to evaluate agent accuracy
2. **Conversation API**: How to programmatically interact with the Genie Agent

The Conversation API is the foundation for:
- Embedding Genie in external applications
- Building custom chatbots powered by Genie
- Automated testing and monitoring
- Slack/Teams integrations


In [ ]:
# Benchmark questions for evaluating the agent
benchmark_questions = [
    # Easy — direct lookups
    "How many active plants do we have?",
    "What product categories do we manufacture?",
    "List all plants in Europe",

    # Medium — aggregations
    "What was total production volume last month?",
    "Which region has the highest yield rate?",
    "What is the average quality pass rate across all plants?",

    # Hard — multi-table joins and complex logic
    "Which plant has the most critical equipment alerts this week?",
    "Compare yield rates between Assembly and Fabrication plants",
    "What is the cost variance trend for the last 3 months?",

    # Persona-specific
    "As an executive, give me a KPI summary for this month",
    "Show me the top 5 suppliers by quality rating",
    "Which products have the highest scrap rate?",
]

# Demo: Ask one question via the Conversation API
if SPACE_ID:
    print("🧪 Testing Conversation API with a sample question...\n")

    # Start a conversation
    conv_payload = {"content": benchmark_questions[0]}
    conv_response = requests.post(
        f"https://{WORKSPACE_URL}/api/2.0/genie/spaces/{SPACE_ID}/conversations",
        headers=headers,
        json=conv_payload
    )

    if conv_response.status_code == 200:
        conv_data = conv_response.json()
        conv_id = conv_data.get("conversation_id", conv_data.get("id"))
        print(f"  ✅ Conversation started: {conv_id}")
        print(f"  📝 Question: {benchmark_questions[0]}")
        print(f"  ⏳ Genie is generating a response...")
        print(f"\n  💡 In a real benchmark, you would:")
        print(f"     1. Send each question via the API")
        print(f"     2. Compare the generated SQL to expected SQL")
        print(f"     3. Compare the result to expected values")
        print(f"     4. Score accuracy and track over time")
    else:
        print(f"  ⚠️ API returned {conv_response.status_code}: {conv_response.text[:200]}")

    print(f"\n📋 Full benchmark set ({len(benchmark_questions)} questions):")
    for i, q in enumerate(benchmark_questions):
        print(f"  {i+1:2d}. {q}")
else:
    print("⚠️ SPACE_ID not set.")
    print(f"\n📋 Benchmark questions prepared ({len(benchmark_questions)}):")
    for i, q in enumerate(benchmark_questions):
        print(f"  {i+1:2d}. {q}")


## 🎉 Cell 7: Summary & Next Steps

### What We Built
- ✅ Genie Agent with **12 tables** attached
- ✅ **18 text instructions** covering business rules, terminology, and defaults
- ✅ **10+ example SQL queries** organized by persona
- ✅ **12 benchmark questions** for accuracy evaluation
- ✅ **Conversation API** demo for programmatic access

### Accuracy Targets
| Metric | Target | How to Improve |
|---|---|---|
| Overall accuracy | >85% | Add more example queries for failed questions |
| Executive questions | >90% | Add verified answers for KPI questions |
| Complex joins | >75% | Add SQL expressions for join patterns |
| Ambiguous terms | >80% | Add more text instructions |

### Next Steps
1. **Open the agent** in the Genie UI and test interactively
2. **Run the full benchmark** and iterate on curation
3. **Proceed to Notebook 3** (Genie One) to configure the business user experience
4. **Proceed to Notebook 4** (Genie Code) for technical practitioner demos


In [ ]:
# Print the agent URL for easy access
if SPACE_ID:
    print(f"🔗 Open your Genie Agent:")
    print(f"   https://{WORKSPACE_URL}/genie/rooms/{SPACE_ID}")
    print(f"\n📱 This agent will also be accessible from:")
    print(f"   • Genie One (web) — auto-routed when users ask manufacturing questions")
    print(f"   • Genie Mobile — same agent, mobile-optimized")
    print(f"   • Genie Code — 'Ask the Manufacturing Operations agent about...'")
    print(f"   • API — POST /api/2.0/genie/spaces/{SPACE_ID}/conversations")
else:
    print("Set SPACE_ID to get the agent URL")

print(f"\n🎉 Notebook 2 complete! Proceed to Notebook 3 (Genie One).")
